# FITS Agent Evaluation — Comparing Agent Strategies for FITS Analysis

Self-contained reproducible evaluation that compares three agent strategies — **ReAct**, **Plan-and-Execute**, and **Reflexion with a symbolic checker** — on FITS-file analysis tasks (metadata extraction, image statistics, anomaly detection). The contribution is replacing Reflexion's natural-language self-critic with a deterministic, rule-based `SymbolicFITSChecker`.

This notebook is self-contained. It bundles all source code (originally split across `agents/`, `tools/`, `oracle/`, `dataset/`, `eval/`, `core/` modules) plus the cached final results in `fits_agent_eval_results/`.

## Pipeline

```
download (MAST) → prepare (ground truth) → run (3 strategies × 3 tasks × N files) → report
```

## Quick start

1. Install the requirements cell (LiteLLM proxy must be reachable).
2. Set env vars (LiteLLM gateway URL + model).
3. Either: (a) jump to **§Final results** to view the cached run, or (b) execute every cell to reproduce.

The cached results were produced on 30 normal + 10 anomaly HST/WFC3 FITS files (MAST public archive), Groq Llama 3.3 70B via LiteLLM, 2 reflection rounds.


## §1 Setup

Install deps and configure the LiteLLM gateway. Skip both cells if you only want to inspect the cached results.

In [ ]:
# Dependencies. astropy + astroquery for FITS I/O and MAST downloads;
# the openai SDK is used to call a LiteLLM-compatible proxy.
%pip install --quiet \
    astropy>=6.0 astroquery>=0.4.7 numpy>=1.26 pandas>=2.1 \
    matplotlib>=3.8 httpx>=0.27 python-dotenv>=1.0 \
    pydantic>=2.6 openai>=1.40 tqdm>=4.66

In [ ]:
import os

# LiteLLM gateway used for all LLM calls. Same proxy as AstroLearn backend.
# Dev default points at Groq via the proxy; swap to claude / gpt by editing
# litellm.yaml — agent code never changes.
os.environ.setdefault("LITELLM_BASE_URL", "http://localhost:4000")
os.environ.setdefault("LITELLM_MODEL", "astrolearn-llm")
os.environ.setdefault("LITELLM_API_KEY", "sk-anything")
os.environ.setdefault("LOG_LEVEL", "INFO")
os.environ.setdefault("RESULTS_DIR", "fits_agent_eval_run/")
os.environ.setdefault("MAX_REFLECTION_ROUNDS", "2")

# Same litellm.yaml the original project used. Start the proxy with:
#   litellm --config litellm.yaml --port 4000
LITELLM_YAML = '''model_list:
  - model_name: astrolearn-llm
    litellm_params:
      model: groq/llama-3.3-70b-versatile
      api_key: os.environ/GROQ_API_KEY

  - model_name: astrolearn-llm-prod
    litellm_params:
      model: anthropic/claude-sonnet-4-6
      api_key: os.environ/ANTHROPIC_API_KEY

litellm_settings:
  drop_params: true
  num_retries: 2
  request_timeout: 120

general_settings:
  master_key: sk-anything
'''
print('OK — env configured. litellm.yaml stored in LITELLM_YAML variable.')

## §2 LLM client (`core/llm_client.py`)

Thin OpenAI-compatible client targeting the LiteLLM proxy. Mirrors AstroLearn's `core/llm/llm_client.py` so anything proven here ports back unchanged.

In [ ]:
"""
Thin LiteLLM gateway client used by all FITS Agent Evaluation agents.

Mirrors AstroLearn's `core/llm/llm_client.py` surface so an agent written here
can be ported into the AstroLearn backend with no interface changes.

Talks to a LiteLLM proxy via OpenAI-compatible chat completions
(URL + model + key come from environment variables).
"""

from __future__ import annotations

import os
import time
from dataclasses import dataclass, field
from typing import Any

from openai import OpenAI


@dataclass
class LLMResponse:
    """Normalized response from a chat completion call."""

    content: str
    tool_calls: list[dict[str, Any]] = field(default_factory=list)
    input_tokens: int = 0
    output_tokens: int = 0
    latency_s: float = 0.0
    raw: Any = None

    @property
    def has_tool_calls(self) -> bool:
        return bool(self.tool_calls)


class LLMClient:
    """OpenAI-compatible client targeting a LiteLLM proxy."""

    def __init__(
        self,
        base_url: str | None = None,
        model: str | None = None,
        api_key: str | None = None,
        temperature: float = 0.2,
        max_tokens: int = 2048,
        timeout: float = 120.0,
    ):
        self.base_url = base_url or os.getenv("LITELLM_BASE_URL", "http://localhost:4000")
        self.model = model or os.getenv("LITELLM_MODEL", "astrolearn-llm")
        self.api_key = api_key or os.getenv("LITELLM_API_KEY", "sk-anything")
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.timeout = timeout
        self._client = OpenAI(
            base_url=self.base_url,
            api_key=self.api_key,
            timeout=timeout,
        )

    def complete(
        self,
        messages: list[dict[str, Any]],
        temperature: float | None = None,
        max_tokens: int | None = None,
    ) -> LLMResponse:
        """Plain text completion without tool calling."""
        t0 = time.time()
        resp = self._client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature if temperature is not None else self.temperature,
            max_tokens=max_tokens or self.max_tokens,
        )
        latency = time.time() - t0
        choice = resp.choices[0].message
        usage = resp.usage
        return LLMResponse(
            content=choice.content or "",
            input_tokens=getattr(usage, "prompt_tokens", 0),
            output_tokens=getattr(usage, "completion_tokens", 0),
            latency_s=latency,
            raw=resp,
        )

    def complete_with_tools(
        self,
        messages: list[dict[str, Any]],
        tools: list[dict[str, Any]],
        temperature: float | None = None,
        max_tokens: int | None = None,
        tool_choice: str = "auto",
    ) -> LLMResponse:
        """Chat completion with OpenAI-style tool definitions."""
        t0 = time.time()
        resp = self._client.chat.completions.create(
            model=self.model,
            messages=messages,
            tools=tools,
            tool_choice=tool_choice,
            temperature=temperature if temperature is not None else self.temperature,
            max_tokens=max_tokens or self.max_tokens,
        )
        latency = time.time() - t0
        choice = resp.choices[0].message
        usage = resp.usage
        # Normalize tool calls to plain dicts so callers don't depend on the SDK shape.
        tool_calls: list[dict[str, Any]] = []
        for tc in choice.tool_calls or []:
            tool_calls.append(
                {
                    "id": tc.id,
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                }
            )
        return LLMResponse(
            content=choice.content or "",
            tool_calls=tool_calls,
            input_tokens=getattr(usage, "prompt_tokens", 0),
            output_tokens=getattr(usage, "completion_tokens", 0),
            latency_s=latency,
            raw=resp,
        )

## §3 Tools — FITS readers (`tools/fits_tools.py`)

`FitsReaderTool` (HDU listing, header read, data stats) and `AstropyTool` (image stats, source detection). Both subclass `BaseTool` which exposes an OpenAI-style tool schema.

In [ ]:
"""
FITS tools for FITS Agent Evaluation agents.

Mirrors the AstroLearn tool interfaces exactly (FitsReaderTool, AstropyTool)
so agents written here can be lifted into the backend without changes.
Backed by real `astropy.io.fits` — not fake data.
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Any

import numpy as np
from astropy.io import fits


class BaseTool(ABC):
    """Strategy base class for FITS Agent Evaluation tools."""

    name: str = ""
    description: str = ""

    @abstractmethod
    def execute(self, fits_path: str, **kwargs: Any) -> dict[str, Any]: ...

    def openai_schema(self) -> dict[str, Any]:
        """Return an OpenAI-compatible tool schema. Override for custom params."""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {},
                    "additionalProperties": True,
                },
            },
        }


def _safe(action: str, fn):
    """Wrap a callable so any exception turns into a structured error dict."""
    try:
        return fn()
    except MemoryError as e:
        return {"error": f"MemoryError: {e}", "action": action}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}", "action": action}


def _with_fits(path: str, fn):
    """Run `fn(hdul)` against an open FITS file. Retries with memmap=False if
    astropy rejects the memmap due to BZERO/BSCALE/BLANK rescaling cards
    (common in raw HST data — the error fires on `.data` access as a
    ValueError, not on open)."""
    try:
        with fits.open(path, memmap=True) as hdul:
            return fn(hdul)
    except (OSError, ValueError) as e:
        msg = str(e)
        if any(k in msg for k in ("BZERO", "BSCALE", "BLANK")):
            with fits.open(path, memmap=False) as hdul:
                return fn(hdul)
        raise


def _hdu_type_name(hdu) -> str:
    return type(hdu).__name__


def _shape_of(hdu) -> list[int]:
    data = getattr(hdu, "data", None)
    if data is None:
        return []
    return list(data.shape)


class FitsReaderTool(BaseTool):
    """Read FITS headers, HDU structure, and basic data statistics."""

    name = "fits_reader"
    description = (
        "Read FITS file headers, HDU structure, and basic data statistics. "
        "Actions: 'list_hdus', 'read_header', 'read_data_stats'."
    )

    def execute(self, fits_path: str, action: str, **kwargs: Any) -> dict[str, Any]:
        if action == "list_hdus":
            return _safe(action, lambda: self._list_hdus(fits_path))
        if action == "read_header":
            hdu_index = int(kwargs.get("hdu_index", 0))
            return _safe(action, lambda: self._read_header(fits_path, hdu_index))
        if action == "read_data_stats":
            hdu_index = int(kwargs.get("hdu_index", 0))
            return _safe(action, lambda: self._read_data_stats(fits_path, hdu_index))
        return {"error": f"Unknown action '{action}'", "action": action}

    def openai_schema(self) -> dict[str, Any]:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "action": {
                            "type": "string",
                            "enum": ["list_hdus", "read_header", "read_data_stats"],
                            "description": "Which read operation to perform.",
                        },
                        "hdu_index": {
                            "type": "integer",
                            "description": "HDU index (required for read_header and read_data_stats).",
                            "default": 0,
                        },
                    },
                    "required": ["action"],
                },
            },
        }

    def _list_hdus(self, fits_path: str) -> dict[str, Any]:
        def work(hdul):
            hdus = []
            for i, hdu in enumerate(hdul):
                hdus.append(
                    {
                        "index": i,
                        "name": hdu.name or "",
                        "type": _hdu_type_name(hdu),
                        "shape": _shape_of(hdu),
                    }
                )
            return {"hdu_count": len(hdul), "hdus": hdus}
        return _with_fits(fits_path, work)

    def _read_header(self, fits_path: str, hdu_index: int) -> dict[str, Any]:
        def work(hdul):
            if hdu_index < 0 or hdu_index >= len(hdul):
                return {"error": f"HDU index {hdu_index} out of range", "action": "read_header"}
            header = hdul[hdu_index].header
            out: dict[str, Any] = {}
            for key in header.keys():
                if not key or key in ("COMMENT", "HISTORY"):
                    continue
                try:
                    val = header[key]
                except Exception:
                    continue
                if isinstance(val, (int, float, bool, str)):
                    out[key] = val
                else:
                    out[key] = str(val)
            return {"header": out}
        return _with_fits(fits_path, work)

    def _read_data_stats(self, fits_path: str, hdu_index: int) -> dict[str, Any]:
        def work(hdul):
            if hdu_index < 0 or hdu_index >= len(hdul):
                return {"error": f"HDU index {hdu_index} out of range", "action": "read_data_stats"}
            data = hdul[hdu_index].data
            if data is None:
                return {"error": "HDU has no data array", "action": "read_data_stats"}
            arr = np.asarray(data, dtype=np.float64)
            nan_ratio = float(np.isnan(arr).mean()) if arr.size else 0.0
            finite = arr[np.isfinite(arr)] if arr.size else arr
            if finite.size == 0:
                return {
                    "mean": float("nan"), "std": float("nan"),
                    "min": float("nan"), "max": float("nan"),
                    "nan_ratio": nan_ratio, "shape": list(arr.shape),
                }
            return {
                "mean": float(np.mean(finite)),
                "std": float(np.std(finite)),
                "min": float(np.min(finite)),
                "max": float(np.max(finite)),
                "nan_ratio": nan_ratio,
                "shape": list(arr.shape),
            }
        return _with_fits(fits_path, work)


class AstropyTool(BaseTool):
    """Compute image statistics and detect bright sources in FITS image HDUs."""

    name = "astropy_tool"
    description = (
        "Compute image statistics and detect bright sources in FITS image HDUs. "
        "Operations: 'image_stats', 'source_detection'."
    )

    def execute(self, fits_path: str, operation: str, **kwargs: Any) -> dict[str, Any]:
        if operation == "image_stats":
            hdu_index = int(kwargs.get("hdu_index", 0))
            return _safe(operation, lambda: self._image_stats(fits_path, hdu_index))
        if operation == "source_detection":
            hdu_index = int(kwargs.get("hdu_index", 0))
            threshold = float(kwargs.get("threshold", 5.0))
            return _safe(
                operation,
                lambda: self._source_detection(fits_path, hdu_index, threshold),
            )
        return {"error": f"Unknown operation '{operation}'", "operation": operation}

    def openai_schema(self) -> dict[str, Any]:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "operation": {
                            "type": "string",
                            "enum": ["image_stats", "source_detection"],
                            "description": "Which computation to run.",
                        },
                        "hdu_index": {"type": "integer", "default": 0},
                        "threshold": {
                            "type": "number",
                            "description": "Sigma threshold for source detection.",
                            "default": 5.0,
                        },
                    },
                    "required": ["operation"],
                },
            },
        }

    def _image_stats(self, fits_path: str, hdu_index: int) -> dict[str, Any]:
        def work(hdul):
            if hdu_index < 0 or hdu_index >= len(hdul):
                return {"error": f"HDU index {hdu_index} out of range", "operation": "image_stats"}
            data = hdul[hdu_index].data
            if data is None:
                return {"error": "HDU has no data array", "operation": "image_stats"}
            arr = np.asarray(data, dtype=np.float64)
            finite = arr[np.isfinite(arr)]
            if finite.size == 0:
                return {
                    "mean": float("nan"), "median": float("nan"),
                    "std": float("nan"), "shape": list(arr.shape),
                }
            return {
                "mean": float(np.mean(finite)),
                "median": float(np.median(finite)),
                "std": float(np.std(finite)),
                "shape": list(arr.shape),
            }
        return _with_fits(fits_path, work)

    def _source_detection(
        self, fits_path: str, hdu_index: int, threshold: float
    ) -> dict[str, Any]:
        """Lightweight peak detection: sigma-threshold + local maxima.

        Avoids photutils dependency — sufficient for benchmark purposes where
        we only need order-of-magnitude source counts to compare strategies.
        """
        def work(hdul):
            if hdu_index < 0 or hdu_index >= len(hdul):
                return {"error": f"HDU index {hdu_index} out of range", "operation": "source_detection"}
            data = hdul[hdu_index].data
            if data is None or data.ndim != 2:
                return {"error": "HDU does not contain a 2D image", "operation": "source_detection"}
            arr = np.asarray(data, dtype=np.float64)
            finite_mask = np.isfinite(arr)
            if not finite_mask.any():
                return {"sources": [], "count": 0}
            mean = float(np.mean(arr[finite_mask]))
            std = float(np.std(arr[finite_mask]))
            cutoff = mean + threshold * std
            mask = (arr > cutoff) & finite_mask
            ys, xs = np.where(mask)
            sources = []
            h, w = arr.shape
            for y, x in zip(ys, xs):
                y0, y1 = max(0, y - 1), min(h, y + 2)
                x0, x1 = max(0, x - 1), min(w, x + 2)
                if arr[y, x] >= arr[y0:y1, x0:x1].max():
                    sources.append({"x": float(x), "y": float(y), "peak": float(arr[y, x])})
                    if len(sources) >= 500:
                        break
            return {"sources": sources, "count": len(sources)}
        return _with_fits(fits_path, work)

## §4 Symbolic checker (`tools/symbolic_checker.py`) — the contribution

`SymbolicFITSChecker` runs 8 deterministic rules (R01–R08) over the FITS structure. Same file → same violations, every run. Replaces the LLM critic in Reflexion.

In [ ]:
"""
Rule-based sanity checker for FITS files — the technical contribution of
this thesis.

Reflexion (Shinn et al. 2023) uses an LLM as its self-critic, which is
prone to hallucination and is non-deterministic. FITS Agent Evaluation replaces that
critic with a *symbolic* checker: a fixed set of deterministic rules over
the FITS structure. Output is a structured `CheckResult` that can be
injected verbatim into the refinement prompt of `ReflexionAgent`.

Reproducibility: same file → same violations, every run.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, Optional

import numpy as np
from astropy.io import fits

# Required WCS keywords for a usable celestial reference frame.
REQUIRED_WCS_KEYS = {"CTYPE1", "CTYPE2", "CRVAL1", "CRVAL2", "CRPIX1", "CRPIX2"}

# Map BITPIX value → expected numpy dtype kind.
BITPIX_KIND = {
    8: "u",   # unsigned int8
    16: "i",
    32: "i",
    64: "i",
    -32: "f",
    -64: "f",
}


@dataclass
class Violation:
    rule_id: str
    severity: str  # "info" | "warning" | "error"
    message: str
    hdu_index: Optional[int] = None

    def to_dict(self) -> dict:
        return {
            "rule_id": self.rule_id,
            "severity": self.severity,
            "message": self.message,
            "hdu_index": self.hdu_index,
        }


@dataclass
class CheckResult:
    passed: bool
    violations: list[Violation] = field(default_factory=list)
    summary: str = ""

    def to_dict(self) -> dict:
        return {
            "passed": self.passed,
            "violations": [v.to_dict() for v in self.violations],
            "summary": self.summary,
        }


# ---------- Individual rule functions ----------
# Each rule inspects an open HDUList and returns a list of Violation (may be empty).

def check_nan_ratio(hdul: fits.HDUList) -> list[Violation]:
    out: list[Violation] = []
    for i, hdu in enumerate(hdul):
        data = getattr(hdu, "data", None)
        if data is None or not hasattr(data, "shape") or data.ndim < 2:
            continue
        arr = np.asarray(data, dtype=np.float64)
        if arr.size == 0:
            continue
        ratio = float(np.isnan(arr).mean())
        if ratio > 0.5:
            out.append(
                Violation(
                    rule_id="R01",
                    severity="warning",
                    message=f"HDU {i} has {ratio:.0%} NaN values — possible data quality issue.",
                    hdu_index=i,
                )
            )
    return out


def check_exptime(hdul: fits.HDUList) -> list[Violation]:
    out: list[Violation] = []
    for i, hdu in enumerate(hdul):
        try:
            if "EXPTIME" not in hdu.header:
                continue
            val = hdu.header["EXPTIME"]
        except Exception:
            continue
        try:
            v = float(val)
        except (TypeError, ValueError):
            continue
        if v <= 0:
            out.append(
                Violation(
                    rule_id="R02",
                    severity="error",
                    message=f"HDU {i} has non-positive EXPTIME={v} — may be calibration frame or invalid.",
                    hdu_index=i,
                )
            )
    return out


def check_empty_hdu(hdul: fits.HDUList) -> list[Violation]:
    out: list[Violation] = []
    for i, hdu in enumerate(hdul):
        data = getattr(hdu, "data", None)
        if data is None:
            continue
        try:
            size = int(np.asarray(data).size)
        except Exception:
            continue
        if size == 0:
            out.append(
                Violation(
                    rule_id="R03",
                    severity="warning",
                    message=f"HDU {i} has zero data elements.",
                    hdu_index=i,
                )
            )
    return out


def check_naxis(hdul: fits.HDUList) -> list[Violation]:
    out: list[Violation] = []
    for i, hdu in enumerate(hdul):
        try:
            naxis = int(hdu.header.get("NAXIS", 0))
        except Exception:
            continue
        data = getattr(hdu, "data", None)
        if data is None:
            if naxis != 0:
                out.append(
                    Violation(
                        rule_id="R04",
                        severity="error",
                        message=f"HDU {i} declares NAXIS={naxis} but has no data array.",
                        hdu_index=i,
                    )
                )
            continue
        if hasattr(data, "ndim") and data.ndim != naxis:
            out.append(
                Violation(
                    rule_id="R04",
                    severity="error",
                    message=f"HDU {i} NAXIS={naxis} does not match data ndim={data.ndim}.",
                    hdu_index=i,
                )
            )
    return out


def check_wcs_keywords(hdul: fits.HDUList) -> list[Violation]:
    out: list[Violation] = []
    for i, hdu in enumerate(hdul):
        data = getattr(hdu, "data", None)
        if data is None or not hasattr(data, "ndim") or data.ndim != 2:
            continue
        present = {k for k in REQUIRED_WCS_KEYS if k in hdu.header}
        missing = REQUIRED_WCS_KEYS - present
        if missing and present:
            out.append(
                Violation(
                    rule_id="R05",
                    severity="info",
                    message=f"HDU {i} has partial WCS — missing {sorted(missing)}.",
                    hdu_index=i,
                )
            )
    return out


def check_bitpix(hdul: fits.HDUList) -> list[Violation]:
    out: list[Violation] = []
    for i, hdu in enumerate(hdul):
        data = getattr(hdu, "data", None)
        if data is None:
            continue
        # Skip BinTable record arrays — BITPIX semantics don't apply.
        if data.dtype.kind == "V":
            continue
        try:
            bitpix = int(hdu.header.get("BITPIX", 0))
        except Exception:
            continue
        expected_kind = BITPIX_KIND.get(bitpix)
        if expected_kind is None:
            continue
        actual_kind = data.dtype.kind
        if actual_kind != expected_kind:
            # Truncate dtype repr — record arrays can produce >5kB strings.
            dtype_str = str(data.dtype)
            if len(dtype_str) > 80:
                dtype_str = dtype_str[:77] + "..."
            out.append(
                Violation(
                    rule_id="R06",
                    severity="warning",
                    message=(
                        f"HDU {i} BITPIX={bitpix} (kind '{expected_kind}') "
                        f"does not match data dtype {dtype_str} (kind '{actual_kind}')."
                    ),
                    hdu_index=i,
                )
            )
    return out


def check_all_zeros(hdul: fits.HDUList) -> list[Violation]:
    out: list[Violation] = []
    for i, hdu in enumerate(hdul):
        data = getattr(hdu, "data", None)
        if data is None or not hasattr(data, "shape") or data.ndim < 2:
            continue
        arr = np.asarray(data)
        if arr.size == 0:
            continue
        finite = arr[np.isfinite(arr)] if arr.dtype.kind == "f" else arr
        if finite.size and np.all(finite == 0):
            out.append(
                Violation(
                    rule_id="R07",
                    severity="warning",
                    message=f"HDU {i} contains all-zero image data.",
                    hdu_index=i,
                )
            )
    return out


def check_header_bloat(hdul: fits.HDUList) -> list[Violation]:
    out: list[Violation] = []
    for i, hdu in enumerate(hdul):
        try:
            n = len(hdu.header)
        except Exception:
            continue
        if n > 1000:
            out.append(
                Violation(
                    rule_id="R08",
                    severity="info",
                    message=f"HDU {i} header has {n} cards (>1000) — possible bloat.",
                    hdu_index=i,
                )
            )
    return out


RuleFn = Callable[[fits.HDUList], list[Violation]]


class SymbolicFITSChecker:
    """Deterministic, rule-based critic used by ReflexionAgent."""

    RULES: list[tuple[str, str, RuleFn, str]] = [
        ("R01", "NaN ratio > 50% in image HDU", check_nan_ratio, "warning"),
        ("R02", "Negative or zero EXPTIME", check_exptime, "error"),
        ("R03", "Empty HDU (zero elements)", check_empty_hdu, "warning"),
        ("R04", "NAXIS mismatch with data shape", check_naxis, "error"),
        ("R05", "Missing required WCS keywords", check_wcs_keywords, "info"),
        ("R06", "BITPIX inconsistent with data type", check_bitpix, "warning"),
        ("R07", "All-zero image data", check_all_zeros, "warning"),
        ("R08", "Header keyword count > 1000", check_header_bloat, "info"),
    ]

    def check(self, fits_path: str) -> CheckResult:
        violations: list[Violation] = []
        try:
            # Some HST raw files reject memmap due to BZERO/BSCALE/BLANK cards.
            try:
                hdul_cm = fits.open(fits_path, memmap=True)
            except Exception:
                hdul_cm = fits.open(fits_path, memmap=False)
            with hdul_cm as hdul:
                for _, _, fn, _ in self.RULES:
                    try:
                        violations.extend(fn(hdul))
                    except (OSError, ValueError) as e:
                        # Retry the file with memmap disabled if BZERO/BSCALE blocked us.
                        if any(k in str(e) for k in ("BZERO", "BSCALE", "BLANK")):
                            with fits.open(fits_path, memmap=False) as hdul2:
                                try:
                                    violations.extend(fn(hdul2))
                                except Exception as e2:
                                    violations.append(
                                        Violation(
                                            rule_id="INTERNAL",
                                            severity="info",
                                            message=f"Rule '{fn.__name__}' failed: {type(e2).__name__}: {e2}",
                                        )
                                    )
                        else:
                            violations.append(
                                Violation(
                                    rule_id="INTERNAL",
                                    severity="info",
                                    message=f"Rule '{fn.__name__}' failed: {type(e).__name__}: {e}",
                                )
                            )
                    except Exception as e:
                        # A single broken rule should not break the whole checker.
                        violations.append(
                            Violation(
                                rule_id="INTERNAL",
                                severity="info",
                                message=f"Rule '{fn.__name__}' failed: {type(e).__name__}: {e}",
                            )
                        )
        except Exception as e:
            return CheckResult(
                passed=False,
                violations=[
                    Violation(
                        rule_id="OPEN",
                        severity="error",
                        message=f"Cannot open FITS file: {type(e).__name__}: {e}",
                    )
                ],
                summary=f"File could not be opened: {e}",
            )

        passed = not any(v.severity == "error" for v in violations)
        return CheckResult(
            passed=passed,
            violations=violations,
            summary=self._summarize(violations),
        )

    @staticmethod
    def _summarize(violations: list[Violation]) -> str:
        if not violations:
            return "No issues detected by symbolic checker."
        # Group sentences by severity so the LLM sees the most important first.
        order = {"error": 0, "warning": 1, "info": 2}
        sorted_v = sorted(violations, key=lambda v: order.get(v.severity, 3))
        lines = [f"[{v.severity.upper()}] {v.message}" for v in sorted_v]
        return " ".join(lines)

## §5 Ground truth oracle (`oracle/ground_truth.py`)

Deterministic answers for all three tasks (T1 metadata / T2 image stats / T3 anomaly) by reading FITS files directly with astropy. Serialized to JSON, consumed by the harness.

In [ ]:
"""
Ground truth oracle for FITS Agent Evaluation.

Produces deterministic answers for all three tasks (T1 metadata, T2 image
stats, T3 anomaly detection) by reading the FITS files directly with
`astropy` and running the `SymbolicFITSChecker`. Output is serialized to
JSON and consumed by the evaluation harness.
"""

from __future__ import annotations

import json
import logging
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
from astropy.io import fits
from pydantic import BaseModel, Field


log = logging.getLogger(__name__)


class MetadataGroundTruth(BaseModel):
    hdu_count: int
    hdu_names: list[str]
    hdu_types: list[str]
    exptime: Optional[float] = None
    telescop: Optional[str] = None
    instrume: Optional[str] = None
    bitpix: int = 0


class ImageStatsGroundTruth(BaseModel):
    hdu_index: int
    mean: float
    std: float
    min: float
    max: float
    nan_ratio: float
    shape: list[int]


class AnomalyGroundTruth(BaseModel):
    has_anomaly: bool
    anomalies: list[str] = Field(default_factory=list)


class GroundTruth(BaseModel):
    file_path: str
    t1: MetadataGroundTruth
    t2: Optional[ImageStatsGroundTruth] = None
    t3: AnomalyGroundTruth


def _round(x: float, ndigits: int = 6) -> float:
    if x is None or not np.isfinite(x):
        return float("nan")
    return float(round(float(x), ndigits))


def _opt_header(header, key: str) -> Optional[str]:
    val = header.get(key)
    if val is None:
        return None
    s = str(val).strip()
    return s or None


def _opt_exptime(header) -> Optional[float]:
    if "EXPTIME" not in header:
        return None
    try:
        return float(header["EXPTIME"])
    except (TypeError, ValueError):
        return None


class GroundTruthGenerator:
    """Generate deterministic ground truth for a FITS file (or a directory)."""

    def __init__(self, checker: SymbolicFITSChecker | None = None):
        self.checker = checker or SymbolicFITSChecker()

    def generate(self, fits_path: str | Path) -> GroundTruth:
        fits_path = str(fits_path)
        try:
            with fits.open(fits_path, memmap=True) as hdul:
                t1 = self._metadata(hdul)
                t2 = self._image_stats(hdul)
        except (OSError, ValueError) as e:
            # Raw HST data with BZERO/BSCALE/BLANK cards rejects memmap when
            # `.data` is accessed; retry with memmap=False.
            if any(k in str(e) for k in ("BZERO", "BSCALE", "BLANK")):
                with fits.open(fits_path, memmap=False) as hdul:
                    t1 = self._metadata(hdul)
                    t2 = self._image_stats(hdul)
            else:
                raise

        check = self.checker.check(fits_path)
        t3 = AnomalyGroundTruth(
            has_anomaly=bool(check.violations),
            anomalies=[v.message for v in check.violations],
        )

        return GroundTruth(file_path=fits_path, t1=t1, t2=t2, t3=t3)

    def generate_batch(self, fits_dir: str | Path) -> list[GroundTruth]:
        fits_dir = Path(fits_dir)
        results: list[GroundTruth] = []
        for path in sorted(fits_dir.rglob("*.fits")):
            try:
                results.append(self.generate(path))
            except Exception as e:
                log.warning("Skipping %s: %s", path, e)
        return results

    def save(self, ground_truths: Iterable[GroundTruth], output_path: str | Path) -> None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        data = [gt.model_dump() for gt in ground_truths]
        with output_path.open("w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        log.info("Wrote %d ground truth records → %s", len(data), output_path)

    @staticmethod
    def load(path: str | Path) -> list[GroundTruth]:
        with Path(path).open("r", encoding="utf-8") as f:
            data = json.load(f)
        return [GroundTruth.model_validate(d) for d in data]

    @staticmethod
    def _metadata(hdul: fits.HDUList) -> MetadataGroundTruth:
        names: list[str] = []
        types: list[str] = []
        for hdu in hdul:
            names.append(hdu.name or "")
            types.append(type(hdu).__name__)
        primary = hdul[0].header
        return MetadataGroundTruth(
            hdu_count=len(hdul),
            hdu_names=names,
            hdu_types=types,
            exptime=_opt_exptime(primary),
            telescop=_opt_header(primary, "TELESCOP"),
            instrume=_opt_header(primary, "INSTRUME"),
            bitpix=int(primary.get("BITPIX", 0) or 0),
        )

    @staticmethod
    def _image_stats(hdul: fits.HDUList) -> Optional[ImageStatsGroundTruth]:
        for i, hdu in enumerate(hdul):
            data = getattr(hdu, "data", None)
            if data is None or not hasattr(data, "ndim") or data.ndim != 2:
                continue
            arr = np.asarray(data, dtype=np.float64)
            nan_ratio = float(np.isnan(arr).mean()) if arr.size else 0.0
            finite = arr[np.isfinite(arr)]
            if finite.size == 0:
                return ImageStatsGroundTruth(
                    hdu_index=i,
                    mean=float("nan"),
                    std=float("nan"),
                    min=float("nan"),
                    max=float("nan"),
                    nan_ratio=_round(nan_ratio),
                    shape=list(arr.shape),
                )
            return ImageStatsGroundTruth(
                hdu_index=i,
                mean=_round(float(np.mean(finite))),
                std=_round(float(np.std(finite))),
                min=_round(float(np.min(finite))),
                max=_round(float(np.max(finite))),
                nan_ratio=_round(nan_ratio),
                shape=list(arr.shape),
            )
        return None

## §6 Dataset downloader (`dataset/download.py`)

Fetches 30 normal (calib_level=3) + 10 anomaly (calib_level=1 raw) HST/WFC3 FITS files from MAST. Idempotent — skips files already on disk.

In [ ]:
"""
Download FITS files from the MAST public archive.

Target dataset for FITS Agent Evaluation:
- 30 "normal" files at calib_level=3 (used for T1 + T2)
- 10 "anomaly" files at calib_level=1 (raw, noisier — used for T3)

`astroquery.mast.Observations` handles the heavy lifting; no MAST token is
required for public data. The function is idempotent — files already
present on disk are skipped.
"""

from __future__ import annotations

import json
import logging
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np
from pydantic import BaseModel
from tqdm import tqdm

log = logging.getLogger(__name__)


class FileEntry(BaseModel):
    filename: str
    path: str
    instrument: Optional[str] = None
    obs_collection: Optional[str] = None
    expected_anomaly: bool = False
    anomaly_type: Optional[str] = None


class DownloadManifest(BaseModel):
    files: list[FileEntry] = []

    def save(self, path: str | Path) -> None:
        Path(path).write_text(self.model_dump_json(indent=2), encoding="utf-8")

    @classmethod
    def load(cls, path: str | Path) -> "DownloadManifest":
        return cls.model_validate_json(Path(path).read_text(encoding="utf-8"))


def _query_observations(
    collection: str,
    instrument: str,
    calib_level: int,
    limit: int,
    t_min: tuple[float, float] = (57000, 57010),
):
    """Constrained MAST query — narrowed by a small MJD window so the result
    set stays manageable (broad queries return ~50k rows and crash
    astroquery's table builder with OOM)."""
    from astroquery.mast import Observations

    obs = Observations.query_criteria(
        obs_collection=collection,
        instrument_name=instrument,
        calib_level=calib_level,
        dataproduct_type="image",
        t_min=list(t_min),
    )
    if obs is None or len(obs) == 0:
        return None
    return obs[: limit * 3]  # over-fetch — many rows yield non-FITS or duplicate products


def _filter_fits_products(products):
    """Keep only SCIENCE products whose filename ends in .fits."""
    is_science = products["productType"] == "SCIENCE"
    # The product table has no `extension` column — check the filename string.
    is_fits = [str(fn).lower().endswith(".fits") for fn in products["productFilename"]]
    return products[is_science & np.array(is_fits)]


def _download_batch(
    products,
    output_dir: Path,
    limit: int,
    collection: str,
    expected_anomaly: bool,
    anomaly_type: Optional[str],
) -> list[FileEntry]:
    from astroquery.mast import Observations

    entries: list[FileEntry] = []
    seen: set[str] = set()
    rows = _filter_fits_products(products)

    with tqdm(total=limit, desc=f"{collection} ({'anomaly' if expected_anomaly else 'normal'})") as pbar:
        for row in rows:
            if len(entries) >= limit:
                break
            uri = row["dataURI"]
            fname = Path(str(uri)).name
            if fname in seen:
                continue
            seen.add(fname)
            target = output_dir / fname
            if target.exists():
                entries.append(
                    FileEntry(
                        filename=fname,
                        path=str(target),
                        instrument=str(row.get("productSubGroupDescription") or ""),
                        obs_collection=collection,
                        expected_anomaly=expected_anomaly,
                        anomaly_type=anomaly_type,
                    )
                )
                pbar.update(1)
                continue
            try:
                result = Observations.download_products(
                    row,
                    download_dir=str(output_dir),
                    cache=True,
                )
                local_paths = list(result["Local Path"]) if result is not None else []
                if not local_paths:
                    continue
                local = Path(local_paths[0])
                # Move into the flat output_dir so we have stable paths.
                if local.exists() and local.parent != output_dir:
                    target = output_dir / local.name
                    if not target.exists():
                        local.replace(target)
                entries.append(
                    FileEntry(
                        filename=target.name,
                        path=str(target),
                        instrument=str(row.get("productSubGroupDescription") or ""),
                        obs_collection=collection,
                        expected_anomaly=expected_anomaly,
                        anomaly_type=anomaly_type,
                    )
                )
                pbar.update(1)
            except Exception as e:
                log.warning("Failed to download %s: %s", fname, e)
                continue
    return entries


def download_dataset(
    output_dir: str | Path = "dataset/fits_files",
    n_normal: int = 30,
    n_anomaly: int = 10,
    calib_level: int = 3,
    manifest_path: str | Path = "dataset/manifest.json",
) -> DownloadManifest:
    """Download a stratified dataset and write a manifest."""
    from astroquery.mast import Observations  # noqa: F401 — fail fast if missing

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    entries: list[FileEntry] = []

    # Normal files — HST/WFC3 at calib_level=3 (calibrated, science-ready).
    # Window (57000, 57010) returns ~624 observations — plenty for 30 files.
    log.info("Querying MAST for %d normal files (HST/WFC3, calib_level=%d)…", n_normal, calib_level)
    obs_norm = _query_observations("HST", "WFC3/UVIS", calib_level, n_normal, t_min=(57000, 57010))
    if obs_norm is not None:
        products = Observations.get_product_list(obs_norm)
        entries.extend(
            _download_batch(
                products,
                output_dir,
                n_normal,
                "HST",
                expected_anomaly=False,
                anomaly_type=None,
            )
        )

    # Anomaly files — raw level-1 frames from a different MJD window where they
    # actually exist (calib_level=1 is sparse; (57400, 57600) yields ~21 obs).
    log.info("Querying MAST for %d anomaly files (HST/WFC3, calib_level=1, raw)…", n_anomaly)
    obs_anom = _query_observations("HST", "WFC3/UVIS", 1, n_anomaly, t_min=(57400, 57600))
    if obs_anom is not None:
        products = Observations.get_product_list(obs_anom)
        entries.extend(
            _download_batch(
                products,
                output_dir,
                n_anomaly,
                "HST",
                expected_anomaly=True,
                anomaly_type="raw_level1",
            )
        )

    manifest = DownloadManifest(files=entries)
    manifest.save(manifest_path)

    # Summary.
    total_size = sum(Path(e.path).stat().st_size for e in entries if Path(e.path).exists())
    by_instrument = Counter(e.instrument or "unknown" for e in entries)
    log.info(
        "Downloaded %d files, %.1f MiB total. By instrument: %s",
        len(entries),
        total_size / (1024 * 1024),
        dict(by_instrument),
    )
    return manifest

## §7 Agents — base + 3 strategies

All three agents subclass `BaseAgent`, return a uniform `AgentResult`, and call tools through the same OpenAI tool-calling protocol. Strategy code below in this order: `base_agent.py` → `react_agent.py` → `plan_execute_agent.py` → `reflexion_agent.py`.

In [ ]:
"""
Abstract base for FITS Agent Evaluation agent strategies.

Interface mirrors AstroLearn's `BaseAgent` so a strategy proven here can
be lifted into the backend with no signature change. All three strategies
(ReAct, Plan-and-Execute, Reflexion) inherit from this class and return
a uniform `AgentResult`.
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Optional



@dataclass
class ToolCallLog:
    tool: str
    input: dict[str, Any]
    output: dict[str, Any]
    latency_ms: int

    def to_dict(self) -> dict[str, Any]:
        return {
            "tool": self.tool,
            "input": self.input,
            "output": self.output,
            "latency_ms": self.latency_ms,
        }


@dataclass
class AgentResult:
    """Uniform result schema across all strategies."""

    answer: str = ""
    extracted_values: dict[str, Any] = field(default_factory=dict)
    tool_calls: list[dict[str, Any]] = field(default_factory=list)
    token_usage: dict[str, int] = field(default_factory=lambda: {"input": 0, "output": 0})
    latency_s: float = 0.0
    error: Optional[str] = None
    reflection_rounds: int = 0
    phase_tokens: dict[str, dict[str, int]] = field(default_factory=dict)

    def to_dict(self) -> dict[str, Any]:
        return {
            "answer": self.answer,
            "extracted_values": self.extracted_values,
            "tool_calls": self.tool_calls,
            "token_usage": self.token_usage,
            "latency_s": self.latency_s,
            "error": self.error,
            "reflection_rounds": self.reflection_rounds,
            "phase_tokens": self.phase_tokens,
        }


class BaseAgent(ABC):
    """Strategy base — orchestrator instantiates concrete agents from this."""

    name: str = "base"

    def __init__(self, tools: list[BaseTool], llm_client: LLMClient):
        self.tools: dict[str, BaseTool] = {t.name: t for t in tools}
        self.llm = llm_client

    @abstractmethod
    def run(self, task: str, fits_path: str) -> AgentResult: ...

    # ---------- helpers shared by subclasses ----------

    def _tool_descriptions(self) -> str:
        return "\n".join(f"- {t.name}: {t.description}" for t in self.tools.values())

    def _openai_tools_schema(self) -> list[dict[str, Any]]:
        return [t.openai_schema() for t in self.tools.values()]

    def _exec_tool(
        self, tool_name: str, fits_path: str, arguments: dict[str, Any]
    ) -> tuple[dict[str, Any], int]:
        """Execute a tool by name and return (output_dict, latency_ms)."""
        import time

        tool = self.tools.get(tool_name)
        if tool is None:
            return {"error": f"Unknown tool '{tool_name}'"}, 0
        t0 = time.time()
        try:
            out = tool.execute(fits_path, **arguments)
        except Exception as e:
            out = {"error": f"{type(e).__name__}: {e}", "tool": tool_name}
        latency_ms = int((time.time() - t0) * 1000)
        return out, latency_ms

In [ ]:
"""
Strategy A — ReAct agent (Yao et al. 2022).

Observe → Think → Act loop, terminating when the LLM stops emitting tool
calls. Capped at MAX_ITERATIONS to prevent infinite loops on
pathological prompts. This is the baseline against which Strategies B
and C are compared.
"""

from __future__ import annotations

import json
import time
from typing import Any


# Tool outputs (especially `read_header` and `list_hdus` on 23-HDU FLC files)
# can balloon past 50kB and crash the LLM's context window. Truncate when
# serializing to the message stream.
MAX_TOOL_OUTPUT_CHARS = 6000


def _serialize_tool_output(out: dict) -> str:
    s = json.dumps(out, default=str)
    if len(s) > MAX_TOOL_OUTPUT_CHARS:
        return s[:MAX_TOOL_OUTPUT_CHARS] + f'... [truncated, {len(s)} chars total]"}}'
    return s


class ReActAgent(BaseAgent):
    name = "react"
    MAX_ITERATIONS = 10

    SYSTEM_PROMPT = (
        "You are an astronomical data analyst.\n"
        "Analyze the given FITS file using the available tools.\n"
        "Think step by step. Call tools to gather information.\n"
        "When you have enough information, provide a complete answer "
        "that explicitly references the values returned by the tools.\n"
        "Available tools:\n{tool_descriptions}\n"
        "FITS file path: {fits_path}\n"
    )

    def run(self, task: str, fits_path: str) -> AgentResult:
        t0 = time.time()
        tool_log: list[dict[str, Any]] = []
        input_tokens = 0
        output_tokens = 0
        error: str | None = None
        final_answer = ""

        messages: list[dict[str, Any]] = [
            {
                "role": "system",
                "content": self.SYSTEM_PROMPT.format(
                    tool_descriptions=self._tool_descriptions(),
                    fits_path=fits_path,
                ),
            },
            {"role": "user", "content": task},
        ]
        tools_schema = self._openai_tools_schema()

        try:
            for _ in range(self.MAX_ITERATIONS):
                resp = self.llm.complete_with_tools(messages, tools=tools_schema)
                input_tokens += resp.input_tokens
                output_tokens += resp.output_tokens

                if not resp.has_tool_calls:
                    final_answer = resp.content
                    break

                # Record the assistant turn with its tool_calls — required so the
                # follow-up tool messages have valid tool_call_id references.
                messages.append(
                    {
                        "role": "assistant",
                        "content": resp.content,
                        "tool_calls": [
                            {
                                "id": tc["id"],
                                "type": "function",
                                "function": {
                                    "name": tc["name"],
                                    "arguments": tc["arguments"],
                                },
                            }
                            for tc in resp.tool_calls
                        ],
                    }
                )

                for tc in resp.tool_calls:
                    try:
                        args = json.loads(tc["arguments"] or "{}")
                    except json.JSONDecodeError:
                        args = {}
                    output, latency_ms = self._exec_tool(tc["name"], fits_path, args)
                    tool_log.append(
                        {
                            "tool": tc["name"],
                            "input": args,
                            "output": output,
                            "latency_ms": latency_ms,
                        }
                    )
                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": tc["id"],
                            "name": tc["name"],
                            "content": _serialize_tool_output(output),
                        }
                    )
            else:
                # Loop exhausted — force a final answer turn without tools.
                resp = self.llm.complete(
                    messages
                    + [
                        {
                            "role": "user",
                            "content": "You have reached the maximum tool-call budget. Provide your best final answer now based on the tool outputs so far.",
                        }
                    ]
                )
                input_tokens += resp.input_tokens
                output_tokens += resp.output_tokens
                final_answer = (resp.content or "").strip() + "\n[Note: max iterations reached]"
        except Exception as e:
            error = f"{type(e).__name__}: {e}"

        return AgentResult(
            answer=final_answer,
            tool_calls=tool_log,
            token_usage={"input": input_tokens, "output": output_tokens},
            latency_s=time.time() - t0,
            error=error,
        )

In [ ]:
"""
Strategy B — Plan-and-Execute agent.

Three phases:
1. PLAN       — LLM emits a JSON execution plan up front.
2. EXECUTE    — Each step in the plan is dispatched to the matching tool.
3. SYNTHESIZE — LLM composes a final answer from the collected outputs.

Trade-off vs. ReAct: the plan is committed up front, so this agent
cannot adapt mid-flight if it encounters an unexpected file structure.
"""

from __future__ import annotations

import json
import re
import time
from typing import Any



def _extract_json(text: str) -> dict | None:
    """Pull the first JSON object out of an LLM response (handles ```json fences)."""
    if not text:
        return None
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    candidate = fenced.group(1) if fenced else None
    if candidate is None:
        # Fallback: first {...} block.
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            candidate = text[start : end + 1]
    if not candidate:
        return None
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        return None


class PlanExecuteAgent(BaseAgent):
    name = "plan_execute"

    PLANNING_PROMPT = (
        "Given this FITS analysis task: {task}\n\n"
        "Generate a JSON execution plan with specific tool calls to make.\n"
        "Return ONLY valid JSON, no explanation:\n"
        "{{\n"
        '    "steps": [\n'
        '        {{"step": 1, "tool": "fits_reader", "action": "list_hdus", "rationale": "..."}},\n'
        '        {{"step": 2, "tool": "fits_reader", "action": "read_header", "hdu_index": 0, "rationale": "..."}}\n'
        "    ]\n"
        "}}\n\n"
        "Available tools and actions:\n"
        "- fits_reader: list_hdus, read_header (needs hdu_index), read_data_stats (needs hdu_index)\n"
        "- astropy_tool: image_stats (needs hdu_index), source_detection (needs hdu_index, optional threshold)\n"
    )

    SYNTHESIS_PROMPT = (
        "You are an astronomical data analyst. The task was:\n{task}\n\n"
        "You executed the following plan against the FITS file at {fits_path}.\n"
        "Tool outputs (in execution order):\n{outputs}\n\n"
        "Write a complete answer to the task. Cite the specific numerical values from "
        "the tool outputs when you make claims, and flag any data-quality issues you notice."
    )

    def run(self, task: str, fits_path: str) -> AgentResult:
        t0 = time.time()
        input_tokens = 0
        output_tokens = 0
        tool_log: list[dict[str, Any]] = []
        error: str | None = None
        answer = ""
        phase_tokens: dict[str, dict[str, int]] = {}

        try:
            # ---- Phase 1: PLAN ----
            plan_resp = self.llm.complete(
                [
                    {"role": "system", "content": "You are a careful planner. Reply with valid JSON only."},
                    {"role": "user", "content": self.PLANNING_PROMPT.format(task=task)},
                ]
            )
            input_tokens += plan_resp.input_tokens
            output_tokens += plan_resp.output_tokens
            phase_tokens["plan"] = {"input": plan_resp.input_tokens, "output": plan_resp.output_tokens}

            plan = _extract_json(plan_resp.content) or {"steps": []}
            steps = plan.get("steps", []) if isinstance(plan, dict) else []

            # ---- Phase 2: EXECUTE ----
            for step in steps:
                if not isinstance(step, dict):
                    continue
                tool_name = step.get("tool")
                if not tool_name or tool_name not in self.tools:
                    tool_log.append(
                        {
                            "tool": str(tool_name),
                            "input": step,
                            "output": {"error": f"Unknown tool '{tool_name}'"},
                            "latency_ms": 0,
                        }
                    )
                    continue
                args = {k: v for k, v in step.items() if k not in ("step", "tool", "rationale")}
                output, latency_ms = self._exec_tool(tool_name, fits_path, args)
                tool_log.append(
                    {
                        "tool": tool_name,
                        "input": args,
                        "output": output,
                        "latency_ms": latency_ms,
                    }
                )

            # ---- Phase 3: SYNTHESIZE ----
            # Cap each per-call output so a 23-HDU header dump doesn't blow context.
            outputs_str = json.dumps(tool_log, indent=2, default=str)
            if len(outputs_str) > 12000:
                outputs_str = outputs_str[:12000] + "\n...[truncated]"
            synth_resp = self.llm.complete(
                [
                    {"role": "system", "content": "You are an astronomical data analyst."},
                    {
                        "role": "user",
                        "content": self.SYNTHESIS_PROMPT.format(
                            task=task, fits_path=fits_path, outputs=outputs_str
                        ),
                    },
                ]
            )
            input_tokens += synth_resp.input_tokens
            output_tokens += synth_resp.output_tokens
            phase_tokens["synthesize"] = {
                "input": synth_resp.input_tokens,
                "output": synth_resp.output_tokens,
            }
            answer = synth_resp.content
        except Exception as e:
            error = f"{type(e).__name__}: {e}"

        return AgentResult(
            answer=answer,
            tool_calls=tool_log,
            token_usage={"input": input_tokens, "output": output_tokens},
            latency_s=time.time() - t0,
            error=error,
            phase_tokens=phase_tokens,
        )

In [ ]:
"""
Strategy C — Reflexion agent with a *symbolic* critic.

This is the main technical contribution of the thesis. The original
Reflexion (Shinn et al. 2023) lets the LLM critique its own answer using
natural language — which is itself hallucination-prone and
non-deterministic. FITS Agent Evaluation replaces that with `SymbolicFITSChecker`:
a rule-based critic whose feedback is deterministic and reproducible.

Loop:
    1. ACT       — Run a full ReAct pass to obtain a v0 answer.
    2. CRITIQUE  — SymbolicFITSChecker inspects the file; an internal
                   deterministic routine checks whether the answer's
                   numeric claims match the tool outputs.
    3. REFINE    — If either critique surfaces issues, re-run ReAct with
                   the structured feedback injected into the prompt.
    4. Capped at MAX_REFLECTIONS to bound cost.
"""

from __future__ import annotations

import json
import os
import re
import time
from typing import Any


NUMBER_RE = re.compile(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?")


def _env_int(name: str, default: int) -> int:
    try:
        return int(os.getenv(name, str(default)))
    except (TypeError, ValueError):
        return default


class ReflexionAgent(BaseAgent):
    name = "reflexion"
    # Env override lets ablation runs change the cap without code edits.
    MAX_REFLECTIONS = _env_int("FITSBENCH_MAX_REFLECTIONS", _env_int("MAX_REFLECTION_ROUNDS", 2))

    REFLECTION_PROMPT = (
        "Your previous analysis of this FITS file:\n{previous_answer}\n\n"
        "Automated checker found the following issues:\n{symbolic_feedback}\n\n"
        "Consistency check against your tool outputs:\n{consistency_issues}\n\n"
        "Please revise your analysis addressing these issues. Be explicit "
        "about any data quality problems you find and cite specific numeric "
        "values from the tool outputs.\n"
        "Original task: {task}\n"
    )

    def __init__(self, tools, llm_client, checker: SymbolicFITSChecker | None = None):
        super().__init__(tools, llm_client)
        self.checker = checker or SymbolicFITSChecker()
        # The inner ReAct agent does the heavy lifting so we don't duplicate the loop.
        self._react = ReActAgent(list(self.tools.values()), llm_client)

    def run(self, task: str, fits_path: str) -> AgentResult:
        t0 = time.time()
        phase_tokens: dict[str, dict[str, int]] = {}

        # ---- 1. ACT: initial answer ----
        result = self._react.run(task, fits_path)
        phase_tokens["act"] = dict(result.token_usage)

        total_in = result.token_usage["input"]
        total_out = result.token_usage["output"]
        rounds_done = 0
        max_reached = False

        for round_idx in range(self.MAX_REFLECTIONS):
            symbolic = self.checker.check(fits_path)
            consistency = self._check_consistency(result)

            phase_tokens.setdefault(f"critique_{round_idx}", {"input": 0, "output": 0})

            if not symbolic.violations and not consistency:
                break

            refined, in_tok, out_tok = self._refine(
                task, fits_path, result, symbolic, consistency, round_idx
            )
            total_in += in_tok
            total_out += out_tok
            phase_tokens[f"refine_{round_idx}"] = {"input": in_tok, "output": out_tok}

            # Merge tool calls so we keep the full trace across rounds.
            refined.tool_calls = result.tool_calls + refined.tool_calls
            result = refined
            rounds_done = round_idx + 1
            if round_idx == self.MAX_REFLECTIONS - 1:
                # If we still have issues after the last allowed round, mark it.
                final_symbolic = self.checker.check(fits_path)
                final_consistency = self._check_consistency(result)
                if final_symbolic.violations or final_consistency:
                    max_reached = True

        if max_reached and "[Note: max reflection rounds reached]" not in result.answer:
            result.answer = (result.answer or "").rstrip() + "\n[Note: max reflection rounds reached]"

        result.token_usage = {"input": total_in, "output": total_out}
        result.latency_s = time.time() - t0
        result.reflection_rounds = rounds_done
        result.phase_tokens = phase_tokens
        return result

    # ---------- Critique helpers ----------

    def _check_consistency(self, result: AgentResult) -> list[str]:
        """Deterministic check that numeric claims in the answer match tool outputs.

        Extracts every number mentioned in the answer text and verifies that
        either (a) it appears in some tool output, or (b) it's a small integer
        (HDU index, count) we don't constrain. Also flags two specific lies:
        - "no NaN" / "0% NaN" claim while a tool reports nan_ratio > 0.05
        - source count mismatch beyond +/- 10%
        """
        issues: list[str] = []
        answer = result.answer or ""
        if not answer:
            return issues

        tool_values: list[float] = []
        nan_ratios: list[float] = []
        source_counts: list[int] = []
        for call in result.tool_calls:
            out = call.get("output") or {}
            if not isinstance(out, dict):
                continue
            for key in ("mean", "std", "min", "max", "median", "nan_ratio"):
                if key in out:
                    try:
                        tool_values.append(float(out[key]))
                    except (TypeError, ValueError):
                        pass
            if "nan_ratio" in out:
                try:
                    nan_ratios.append(float(out["nan_ratio"]))
                except (TypeError, ValueError):
                    pass
            if "count" in out and isinstance(out["count"], int):
                source_counts.append(out["count"])

        # Check 1: NaN denial.
        lower = answer.lower()
        nan_denial = any(
            phrase in lower for phrase in ("no nan", "0% nan", "no missing data", "zero nan")
        )
        if nan_denial and any(r > 0.05 for r in nan_ratios):
            issues.append(
                f"Answer claims no NaN values, but tool output reports nan_ratio up to "
                f"{max(nan_ratios):.2%}."
            )

        # Check 2: numbers in answer that don't trace back to any tool output.
        answer_numbers = [float(m) for m in NUMBER_RE.findall(answer)]
        ungrounded: list[float] = []
        for n in answer_numbers:
            if abs(n) < 1e-9:
                continue
            if any(_close(n, v) for v in tool_values):
                continue
            if n.is_integer() and abs(n) < 1000:
                # Small integers: assume HDU indices / counts — too noisy to flag.
                continue
            ungrounded.append(n)
        if ungrounded:
            shown = ", ".join(f"{n:g}" for n in ungrounded[:5])
            issues.append(
                f"Answer mentions numeric values that do not match any tool output: {shown}."
            )

        # Check 3: source-count mismatch.
        if source_counts:
            for n in answer_numbers:
                if n.is_integer() and 0 < n < 10_000:
                    actual = max(source_counts)
                    if actual > 0 and abs(n - actual) / max(actual, 1) > 0.10 and abs(n - actual) > 3:
                        # Heuristic: only flag if the answer's number is close to the
                        # right order of magnitude — avoids flagging unrelated ints.
                        if 0.1 <= n / max(actual, 1) <= 10:
                            issues.append(
                                f"Answer mentions {int(n)} sources but source_detection found {actual}."
                            )
                            break

        return issues

    def _refine(
        self,
        task: str,
        fits_path: str,
        previous: AgentResult,
        symbolic: CheckResult,
        consistency: list[str],
        round_idx: int,
    ) -> tuple[AgentResult, int, int]:
        """Run a fresh ReAct pass with the critique feedback injected."""
        feedback = symbolic.summary or "(no symbolic violations)"
        cons_text = "; ".join(consistency) if consistency else "(no consistency issues)"
        augmented_task = self.REFLECTION_PROMPT.format(
            previous_answer=previous.answer,
            symbolic_feedback=feedback,
            consistency_issues=cons_text,
            task=task,
        )
        refined = self._react.run(augmented_task, fits_path)
        return refined, refined.token_usage["input"], refined.token_usage["output"]


def _close(a: float, b: float, rel: float = 1e-3, abs_tol: float = 1e-6) -> bool:
    if a == b:
        return True
    return abs(a - b) <= max(abs_tol, rel * max(abs(a), abs(b)))

## §8 Metrics (`eval/metrics.py`)

Deterministic scoring:
- **T1** per-field exact-match on metadata.
- **T2** MAPE across mean/std/min/max (<1% = correct, <5% = partial).
- **T3** binary F1 on `has_anomaly` + type-level F1.
- **AstroExplain** 4-criterion rubric (E1 precision / E2 uncertainty / E3 traceability / E4 anomaly flag, each +0.25).

In [ ]:
"""
Metrics for FITS Agent Evaluation.

- T1 (metadata extraction): per-field exact-match.
- T2 (image stats):         MAPE across mean/std/min/max with tolerance bands.
- T3 (anomaly detection):   binary F1 on has_anomaly; F1 on anomaly types if available.
- AstroExplain Score:       deterministic 4-criterion rubric (no LLM needed).
"""

from __future__ import annotations

import math
import re
from dataclasses import dataclass, field
from typing import Any

    AnomalyGroundTruth,
    ImageStatsGroundTruth,
    MetadataGroundTruth,
)

# ---------- score dataclasses ----------


@dataclass
class T1Score:
    field_scores: dict[str, float]
    overall: float
    matched: int
    total: int

    def to_dict(self) -> dict[str, Any]:
        return self.__dict__


@dataclass
class T2Score:
    field_mape: dict[str, float]
    overall_mape: float
    band: str  # "correct" | "partial" | "incorrect"

    def to_dict(self) -> dict[str, Any]:
        return self.__dict__


@dataclass
class T3Score:
    has_anomaly_correct: bool
    type_precision: float
    type_recall: float
    type_f1: float

    def to_dict(self) -> dict[str, Any]:
        return self.__dict__


@dataclass
class AstroExplainScore:
    e1_precision: float
    e2_uncertainty: float
    e3_traceability: float
    e4_anomaly_flag: float
    total: float
    notes: dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> dict[str, Any]:
        return self.__dict__


# ---------- helpers ----------

NUMBER_RE = re.compile(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?")

UNCERTAINTY_KEYWORDS = (
    "may", "might", "appears to", "approximately", "around ", "roughly",
    "uncertain", "unclear", "possibly", "likely", "seems", "estimated",
    "approx.", "~", "could be",
)


def _norm_str(s: Any) -> str:
    if s is None:
        return ""
    return str(s).strip().lower()


def _safe_eq(a: Any, b: Any) -> bool:
    if a is None and b is None:
        return True
    if a is None or b is None:
        return False
    return _norm_str(a) == _norm_str(b)


def _safe_eq_list(a: Any, b: Any) -> bool:
    if a is None and b is None:
        return True
    if a is None or b is None:
        return False
    if not (isinstance(a, list) and isinstance(b, list)):
        return False
    return [_norm_str(x) for x in a] == [_norm_str(x) for x in b]


def _mape(predicted: float, truth: float) -> float:
    if truth is None or predicted is None:
        return float("inf")
    if not (math.isfinite(predicted) and math.isfinite(truth)):
        return float("inf")
    if truth == 0:
        return 0.0 if predicted == 0 else float("inf")
    return abs(predicted - truth) / abs(truth)


# ---------- metrics ----------


class BenchmarkMetrics:
    """Static methods for computing all benchmark scores."""

    # ----- T1 -----
    @staticmethod
    def t1_exact_match(predicted: dict, ground_truth: MetadataGroundTruth) -> T1Score:
        fields = {
            "hdu_count": _safe_eq(predicted.get("hdu_count"), ground_truth.hdu_count),
            "hdu_names": _safe_eq_list(predicted.get("hdu_names"), ground_truth.hdu_names),
            "hdu_types": _safe_eq_list(predicted.get("hdu_types"), ground_truth.hdu_types),
            "exptime": _safe_eq(predicted.get("exptime"), ground_truth.exptime),
            "telescop": _safe_eq(predicted.get("telescop"), ground_truth.telescop),
            "instrume": _safe_eq(predicted.get("instrume"), ground_truth.instrume),
            "bitpix": _safe_eq(predicted.get("bitpix"), ground_truth.bitpix),
        }
        scores = {k: (1.0 if v else 0.0) for k, v in fields.items()}
        matched = sum(1 for v in fields.values() if v)
        total = len(fields)
        return T1Score(
            field_scores=scores,
            overall=matched / total if total else 0.0,
            matched=matched,
            total=total,
        )

    # ----- T2 -----
    @staticmethod
    def t2_mape(predicted: dict, ground_truth: ImageStatsGroundTruth) -> T2Score:
        keys = ("mean", "std", "min", "max")
        field_mape: dict[str, float] = {}
        for k in keys:
            field_mape[k] = _mape(_to_float(predicted.get(k)), getattr(ground_truth, k))
        finite_vals = [v for v in field_mape.values() if math.isfinite(v)]
        overall = sum(finite_vals) / len(finite_vals) if finite_vals else float("inf")
        if overall < 0.01:
            band = "correct"
        elif overall < 0.05:
            band = "partial"
        else:
            band = "incorrect"
        return T2Score(field_mape=field_mape, overall_mape=overall, band=band)

    # ----- T3 -----
    @staticmethod
    def t3_f1(predicted_anomalies: list[str], ground_truth: AnomalyGroundTruth) -> T3Score:
        pred_has = bool(predicted_anomalies)
        gt_has = bool(ground_truth.has_anomaly)
        has_anomaly_correct = pred_has == gt_has

        gt_set = {_norm_str(a) for a in ground_truth.anomalies}
        pred_set = {_norm_str(a) for a in predicted_anomalies}

        if not gt_set and not pred_set:
            return T3Score(has_anomaly_correct=has_anomaly_correct, type_precision=1.0, type_recall=1.0, type_f1=1.0)
        if not pred_set:
            return T3Score(has_anomaly_correct=has_anomaly_correct, type_precision=0.0, type_recall=0.0, type_f1=0.0)
        if not gt_set:
            return T3Score(has_anomaly_correct=has_anomaly_correct, type_precision=0.0, type_recall=0.0, type_f1=0.0)

        # Substring match — predicted anomaly mentions the gt anomaly text (or vice versa).
        true_positives = 0
        for p in pred_set:
            if any(p in g or g in p for g in gt_set):
                true_positives += 1
        precision = true_positives / len(pred_set) if pred_set else 0.0
        recall = true_positives / len(gt_set) if gt_set else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        return T3Score(
            has_anomaly_correct=has_anomaly_correct,
            type_precision=precision,
            type_recall=recall,
            type_f1=f1,
        )

    # ----- AstroExplain -----
    @staticmethod
    def astro_explain_score(
        answer: str,
        tool_calls: list[dict],
        nan_ratio: float | None = None,
        has_anomaly: bool = False,
    ) -> AstroExplainScore:
        """4-criterion rubric, each worth +0.25, total ∈ [0, 1]."""
        answer = answer or ""
        lower = answer.lower()
        notes: dict[str, Any] = {}

        # E1 — Precision: explicit HDU index or header keyword reference.
        e1 = 0.0
        if re.search(r"hdu\s*\d+|hdu\s*index", lower) or re.search(
            r"\b(exptime|naxis|bitpix|telescop|instrume|ctype1|crval1)\b", lower
        ):
            e1 = 0.25
        notes["e1_match"] = bool(e1)

        # E2 — Uncertainty: hedging language when nan_ratio > 0.1.
        e2 = 0.0
        if nan_ratio is not None and nan_ratio > 0.1:
            if any(kw in lower for kw in UNCERTAINTY_KEYWORDS):
                e2 = 0.25
        else:
            # No high-NaN penalty applicable — give credit by default.
            e2 = 0.25
        notes["e2_high_nan_required"] = bool(nan_ratio is not None and nan_ratio > 0.1)

        # E3 — Traceability: at least one numeric value from a tool output appears in the answer.
        tool_numbers = BenchmarkMetrics._collect_tool_numbers(tool_calls)
        answer_numbers = [float(m) for m in NUMBER_RE.findall(answer)]
        traced = 0
        for n in answer_numbers:
            if any(_close(n, t) for t in tool_numbers):
                traced += 1
        e3 = 0.25 if traced > 0 else 0.0
        notes["e3_traced_count"] = traced

        # E4 — Anomaly flag: if file has anomaly, the answer must mention it.
        e4 = 0.0
        if has_anomaly:
            anomaly_words = ("nan", "anomaly", "anomalous", "missing", "issue", "problem",
                             "saturat", "negative exptime", "all-zero", "all zero", "calibration",
                             "data quality", "outlier", "invalid")
            if any(w in lower for w in anomaly_words):
                e4 = 0.25
        else:
            e4 = 0.25  # nothing to flag → full credit.

        total = e1 + e2 + e3 + e4
        return AstroExplainScore(
            e1_precision=e1,
            e2_uncertainty=e2,
            e3_traceability=e3,
            e4_anomaly_flag=e4,
            total=total,
            notes=notes,
        )

    @staticmethod
    def _collect_tool_numbers(tool_calls: list[dict]) -> list[float]:
        nums: list[float] = []
        for call in tool_calls or []:
            out = call.get("output") or {}
            if isinstance(out, dict):
                nums.extend(_extract_numbers(out))
        return nums


def _to_float(x: Any) -> float | None:
    if x is None:
        return None
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


def _close(a: float, b: float, rel: float = 5e-3, abs_tol: float = 1e-6) -> bool:
    if a == b:
        return True
    if not (math.isfinite(a) and math.isfinite(b)):
        return False
    return abs(a - b) <= max(abs_tol, rel * max(abs(a), abs(b)))


def _extract_numbers(obj: Any) -> list[float]:
    out: list[float] = []
    if isinstance(obj, (int, float)) and not isinstance(obj, bool):
        if math.isfinite(float(obj)):
            out.append(float(obj))
    elif isinstance(obj, dict):
        for v in obj.values():
            out.extend(_extract_numbers(v))
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            out.extend(_extract_numbers(v))
    return out

## §9 Harness (`eval/harness.py`)

Drives N files × M strategies × 3 tasks, parses each agent's tool log into a predicted answer, scores it against the ground truth, writes `results.csv`.

In [ ]:
"""
End-to-end benchmark harness.

Runs N files × M strategies × 3 task types. Each row in the resulting
CSV captures the per-task scores plus efficiency metrics (tool calls,
token usage, latency, reflection rounds). Designed to be called both
from `run_benchmark.py` (CLI) and from notebooks.
"""

from __future__ import annotations

import json
import logging
import re
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

import pandas as pd
from tqdm import tqdm


log = logging.getLogger(__name__)


TASK_PROMPTS = {
    "t1_metadata": (
        "Extract the FITS file metadata: total HDU count, HDU names and types, "
        "BITPIX, EXPTIME, TELESCOP, INSTRUME. Report each value clearly."
    ),
    "t2_image_stats": (
        "Compute the image statistics (mean, std, min, max, nan_ratio, shape) "
        "for the first 2-D image HDU in the file. Report which HDU you used."
    ),
    "t3_anomaly": (
        "Inspect the file for data-quality anomalies (NaN, bad EXPTIME, empty HDUs, "
        "NAXIS / BITPIX inconsistencies, all-zero data, missing WCS). List any issues "
        "you find with the HDU index."
    ),
}


@dataclass
class BenchmarkRow:
    file_id: str
    strategy: str
    task_type: str
    t1_score: Optional[float] = None
    t2_mape: Optional[float] = None
    t3_f1: Optional[float] = None
    astro_explain: Optional[float] = None
    tool_call_count: int = 0
    token_input: int = 0
    token_output: int = 0
    latency_s: float = 0.0
    reflection_rounds: int = 0
    error: Optional[str] = None
    raw_answer: str = ""


@dataclass
class BenchmarkResults:
    rows: list[BenchmarkRow]
    dataframe: pd.DataFrame = field(default_factory=pd.DataFrame)


# ---------- answer parsing ----------

NUMBER_RE = re.compile(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?")


def _last_tool_output(tool_calls: list[dict], tool_name: str, **match) -> dict | None:
    """Return the last tool_call output matching tool_name + input filters."""
    for call in reversed(tool_calls):
        if call.get("tool") != tool_name:
            continue
        inp = call.get("input") or {}
        if all(inp.get(k) == v for k, v in match.items()):
            out = call.get("output")
            if isinstance(out, dict) and "error" not in out:
                return out
    return None


def _extract_t1_values(result: AgentResult) -> dict[str, Any]:
    """Build a predicted T1 dict from the agent's tool calls (deterministic)."""
    pred: dict[str, Any] = {}
    list_hdus_out = _last_tool_output(result.tool_calls, "fits_reader", action="list_hdus")
    if list_hdus_out:
        pred["hdu_count"] = list_hdus_out.get("hdu_count")
        hdus = list_hdus_out.get("hdus") or []
        pred["hdu_names"] = [h.get("name", "") for h in hdus]
        pred["hdu_types"] = [h.get("type", "") for h in hdus]
    header_out = _last_tool_output(result.tool_calls, "fits_reader", action="read_header", hdu_index=0)
    if header_out and isinstance(header_out.get("header"), dict):
        header = header_out["header"]
        pred["exptime"] = header.get("EXPTIME")
        pred["telescop"] = header.get("TELESCOP")
        pred["instrume"] = header.get("INSTRUME")
        pred["bitpix"] = header.get("BITPIX")
    return pred


def _extract_t2_values(result: AgentResult, hdu_index: int) -> dict[str, Any]:
    out = (
        _last_tool_output(result.tool_calls, "fits_reader", action="read_data_stats", hdu_index=hdu_index)
        or _last_tool_output(result.tool_calls, "astropy_tool", operation="image_stats", hdu_index=hdu_index)
    )
    if not out:
        # Fall back to any data-stats call.
        for call in reversed(result.tool_calls):
            if call.get("tool") == "fits_reader" and (call.get("input") or {}).get("action") == "read_data_stats":
                cand = call.get("output")
                if isinstance(cand, dict) and "error" not in cand:
                    out = cand
                    break
    if not out:
        return {}
    return {
        "mean": out.get("mean"),
        "std": out.get("std"),
        "min": out.get("min"),
        "max": out.get("max"),
        "nan_ratio": out.get("nan_ratio"),
        "shape": out.get("shape"),
    }


def _extract_t3_anomalies(result: AgentResult, answer: str) -> list[str]:
    """Heuristic: split the answer into sentences and keep ones that look like anomaly mentions."""
    anomaly_keywords = (
        "nan", "anomal", "missing", "invalid", "negative exptime", "zero exptime",
        "all-zero", "all zero", "naxis mismatch", "bitpix mismatch", "missing wcs",
        "empty hdu", "bloated header",
    )
    sentences = re.split(r"(?<=[.!?])\s+", answer or "")
    found = [s.strip() for s in sentences if any(k in s.lower() for k in anomaly_keywords)]
    return found


# ---------- harness ----------


class BenchmarkHarness:
    def __init__(self, llm_client: LLMClient | None = None):
        self.llm = llm_client or LLMClient()
        self.checker = SymbolicFITSChecker()
        tools = [FitsReaderTool(), AstropyTool()]
        self.agents: dict[str, BaseAgent] = {
            "react": ReActAgent(tools, self.llm),
            "plan_execute": PlanExecuteAgent(tools, self.llm),
            "reflexion": ReflexionAgent(tools, self.llm, self.checker),
        }

    def run(
        self,
        dataset_dir: str | Path,
        ground_truth_path: str | Path,
        output_dir: str | Path = "results/",
        strategies: list[str] | None = None,
        verbose: bool = True,
    ) -> BenchmarkResults:
        strategies = strategies or list(self.agents.keys())
        ground_truths = GroundTruthGenerator.load(ground_truth_path)
        log.info("Loaded %d ground truth records.", len(ground_truths))

        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        rows: list[BenchmarkRow] = []
        for gt in tqdm(ground_truths, desc="files", disable=not verbose):
            for strat in strategies:
                agent = self.agents.get(strat)
                if agent is None:
                    log.warning("Unknown strategy '%s' — skipping", strat)
                    continue
                for task_type, prompt in TASK_PROMPTS.items():
                    rows.append(self._run_single(agent, gt, strat, task_type, prompt))

        df = self._to_dataframe(rows)
        csv_path = output_dir / "results.csv"
        df.to_csv(csv_path, index=False)
        log.info("Wrote results CSV → %s", csv_path)

        if verbose:
            self._print_summary(df)
        return BenchmarkResults(rows=rows, dataframe=df)

    def _run_single(
        self,
        agent: BaseAgent,
        gt: GroundTruth,
        strategy: str,
        task_type: str,
        prompt: str,
    ) -> BenchmarkRow:
        t0 = time.time()
        try:
            result = agent.run(prompt, gt.file_path)
        except Exception as e:
            return BenchmarkRow(
                file_id=Path(gt.file_path).name,
                strategy=strategy,
                task_type=task_type,
                error=f"{type(e).__name__}: {e}",
                latency_s=time.time() - t0,
            )

        row = BenchmarkRow(
            file_id=Path(gt.file_path).name,
            strategy=strategy,
            task_type=task_type,
            tool_call_count=len(result.tool_calls),
            token_input=result.token_usage.get("input", 0),
            token_output=result.token_usage.get("output", 0),
            latency_s=result.latency_s,
            reflection_rounds=result.reflection_rounds,
            error=result.error,
            raw_answer=result.answer or "",
        )

        # Score the task this row represents.
        if task_type == "t1_metadata":
            pred = _extract_t1_values(result)
            score = BenchmarkMetrics.t1_exact_match(pred, gt.t1)
            row.t1_score = score.overall
        elif task_type == "t2_image_stats" and gt.t2 is not None:
            pred = _extract_t2_values(result, gt.t2.hdu_index)
            score = BenchmarkMetrics.t2_mape(pred, gt.t2)
            row.t2_mape = score.overall_mape
        elif task_type == "t3_anomaly":
            anomalies = _extract_t3_anomalies(result, result.answer or "")
            score = BenchmarkMetrics.t3_f1(anomalies, gt.t3)
            row.t3_f1 = score.type_f1

        # AstroExplain runs on every task — it grades the explanation, not the values.
        explain = BenchmarkMetrics.astro_explain_score(
            answer=result.answer or "",
            tool_calls=result.tool_calls,
            nan_ratio=(gt.t2.nan_ratio if gt.t2 else None),
            has_anomaly=gt.t3.has_anomaly,
        )
        row.astro_explain = explain.total
        return row

    @staticmethod
    def _to_dataframe(rows: list[BenchmarkRow]) -> pd.DataFrame:
        return pd.DataFrame([r.__dict__ for r in rows])

    @staticmethod
    def _print_summary(df: pd.DataFrame) -> None:
        if df.empty:
            print("(no results)")
            return
        agg = df.groupby("strategy").agg(
            t1_mean=("t1_score", "mean"),
            t2_mean=("t2_mape", "mean"),
            t3_mean=("t3_f1", "mean"),
            explain_mean=("astro_explain", "mean"),
            tool_calls_mean=("tool_call_count", "mean"),
            tokens_mean=("token_input", "mean"),
            latency_mean=("latency_s", "mean"),
            reflection_mean=("reflection_rounds", "mean"),
        )
        print("\n=== Summary by strategy ===")
        print(agg.round(3).to_string())

## §10 Post-hoc analysis (`eval/analysis.py`)

Reconstructs the AstroExplain E1/E2/E3/E4 breakdown from `raw_answer`, renders the four thesis figures, computes the ablation curve over `MAX_REFLECTIONS ∈ {1, 2, 3}`.

In [ ]:
"""
Post-hoc analysis script that consumes results.csv + ground_truth.json and
produces the figures + tables needed for the thesis Results chapter:

1. AstroExplain E1/E2/E3/E4 component breakdown per strategy.
2. Correlation between T3 F1 (Reflexion) and per-file anomaly_count.
3. Box plot of T3 F1 distribution per strategy.
4. Scatter plot of anomaly_count vs T3 F1 for Reflexion.
5. Ablation curve: MAX_REFLECTIONS in {1, 2, 3} vs T3 F1 (mean).

Re-computes E1/E2/E4 from `raw_answer` text + ground truth (deterministic).
E3 (traceability) is recovered as `astro_explain_total - (E1 + E2 + E4)`
since each component is exactly 0 or 0.25, so we don't need the original
tool_calls list.
"""

from __future__ import annotations

import argparse
import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))



def _load_ground_truth(path: Path) -> dict:
    data = json.loads(path.read_text(encoding="utf-8"))
    return {Path(d["file_path"]).name: d for d in data}


def _e1_precision(answer: str) -> float:
    lower = (answer or "").lower()
    if re.search(r"hdu\s*\d+|hdu\s*index", lower) or re.search(
        r"\b(exptime|naxis|bitpix|telescop|instrume|ctype1|crval1)\b", lower
    ):
        return 0.25
    return 0.0


def _e2_uncertainty(answer: str, nan_ratio: float | None) -> float:
    if nan_ratio is None or nan_ratio <= 0.1:
        return 0.25  # nothing to hedge about → full credit
    lower = (answer or "").lower()
    return 0.25 if any(kw in lower for kw in UNCERTAINTY_KEYWORDS) else 0.0


def _e4_anomaly_flag(answer: str, has_anomaly: bool) -> float:
    if not has_anomaly:
        return 0.25
    lower = (answer or "").lower()
    keywords = (
        "nan", "anomaly", "anomalous", "missing", "issue", "problem",
        "saturat", "negative exptime", "all-zero", "all zero", "calibration",
        "data quality", "outlier", "invalid",
    )
    return 0.25 if any(k in lower for k in keywords) else 0.0


def attach_e_components(df: pd.DataFrame, gt: dict) -> pd.DataFrame:
    """Add E1/E2/E3/E4 columns by re-scoring deterministically."""
    e1, e2, e3, e4 = [], [], [], []
    for _, row in df.iterrows():
        meta = gt.get(row["file_id"])
        if meta is None:
            e1.append(np.nan); e2.append(np.nan); e3.append(np.nan); e4.append(np.nan)
            continue
        nan_ratio = (meta.get("t2") or {}).get("nan_ratio")
        has_anomaly = bool(meta["t3"]["has_anomaly"])
        ans = str(row.get("raw_answer", ""))
        v1 = _e1_precision(ans)
        v2 = _e2_uncertainty(ans, nan_ratio)
        v4 = _e4_anomaly_flag(ans, has_anomaly)
        total = float(row.get("astro_explain", 0.0) or 0.0)
        v3 = max(0.0, total - v1 - v2 - v4)
        # Snap to 0/0.25 since each component is binary.
        v3 = 0.25 if v3 >= 0.125 else 0.0
        e1.append(v1); e2.append(v2); e3.append(v3); e4.append(v4)
    df = df.copy()
    df["e1_precision"] = e1
    df["e2_uncertainty"] = e2
    df["e3_traceability"] = e3
    df["e4_anomaly_flag"] = e4
    return df


# ---------- Figures ----------

def fig_components_bar(df: pd.DataFrame, out: Path) -> None:
    by_strategy = df.groupby("strategy")[["e1_precision", "e2_uncertainty", "e3_traceability", "e4_anomaly_flag"]].mean()
    by_strategy = by_strategy.loc[["react", "plan_execute", "reflexion"]]
    ax = by_strategy.plot(
        kind="bar",
        figsize=(9, 5),
        title="AstroExplain components per strategy (mean over 135 rows)",
    )
    ax.set_ylabel("Mean component score (0 or 0.25)")
    ax.set_ylim(0, 0.27)
    ax.set_xticklabels(by_strategy.index, rotation=0)
    ax.legend(["E1 precision", "E2 uncertainty", "E3 traceability", "E4 anomaly flag"])
    plt.tight_layout()
    plt.savefig(out, dpi=130)
    plt.close()


def fig_t3_box(df: pd.DataFrame, out: Path) -> None:
    t3 = df[df["task_type"] == "t3_anomaly"]
    data = [t3[t3["strategy"] == s]["t3_f1"].dropna().values for s in ["react", "plan_execute", "reflexion"]]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.boxplot(data, labels=["ReAct", "Plan-Execute", "Reflexion"], showmeans=True)
    ax.set_ylabel("T3 F1")
    ax.set_title("T3 anomaly detection F1 — distribution across 45 files")
    ax.set_ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.savefig(out, dpi=130)
    plt.close()


def fig_scatter_anomaly(df: pd.DataFrame, gt: dict, out: Path) -> tuple[float, float]:
    """Scatter: anomaly_count (from ground truth) vs T3 F1 (Reflexion).
    Returns (Pearson r, Spearman rho)."""
    refl_t3 = df[(df["strategy"] == "reflexion") & (df["task_type"] == "t3_anomaly")].copy()
    refl_t3["anomaly_count"] = refl_t3["file_id"].map(
        lambda f: len(gt.get(f, {}).get("t3", {}).get("anomalies", []))
    )
    x = refl_t3["anomaly_count"].astype(float).values
    y = refl_t3["t3_f1"].astype(float).values
    if len(x) < 2:
        return (float("nan"), float("nan"))
    pearson = float(np.corrcoef(x, y)[0, 1])
    # Spearman = Pearson on ranks. Avoids scipy dependency.
    rx = pd.Series(x).rank().values
    ry = pd.Series(y).rank().values
    spearman = float(np.corrcoef(rx, ry)[0, 1])
    fig, ax = plt.subplots(figsize=(7, 5))
    jitter = (np.random.RandomState(0).rand(len(x)) - 0.5) * 0.15
    ax.scatter(x + jitter, y, alpha=0.7, s=70)
    if len(x) >= 2 and np.std(x) > 0:
        coef = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 100)
        ax.plot(xs, np.polyval(coef, xs), "r--", label=f"linear fit (r={pearson:.3f})")
        ax.legend()
    ax.set_xlabel("Anomaly count (ground truth)")
    ax.set_ylabel("T3 F1 (Reflexion)")
    ax.set_title(f"Reflexion T3 F1 vs anomaly count — Pearson r={pearson:.3f}, Spearman ρ={spearman:.3f}")
    plt.tight_layout()
    plt.savefig(out, dpi=130)
    plt.close()
    return pearson, spearman


def fig_ablation_curve(values: dict[int, float], out: Path) -> None:
    """Plot mean T3 F1 against MAX_REFLECTIONS setting."""
    xs = sorted(values.keys())
    ys = [values[k] for k in xs]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(xs, ys, "o-", linewidth=2, markersize=10)
    for x, y in zip(xs, ys):
        ax.annotate(f"{y:.3f}", (x, y), xytext=(8, 8), textcoords="offset points")
    ax.set_xlabel("MAX_REFLECTIONS")
    ax.set_ylabel("Reflexion mean T3 F1")
    ax.set_title("Ablation A — reflection budget vs T3 anomaly F1 (45 files)")
    ax.set_xticks(xs)
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(out, dpi=130)
    plt.close()


# ---------- Main ----------

def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--results", default="results/results.csv")
    parser.add_argument("--ground-truth", default="dataset/ground_truth.json")
    parser.add_argument("--out-dir", default="results")
    parser.add_argument(
        "--ablation",
        nargs="*",
        default=[],
        help="Pairs MAX:csv (e.g. 1:results/ablation_max1/results.csv) for the ablation curve.",
    )
    args = parser.parse_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    gt = _load_ground_truth(Path(args.ground_truth))
    df = pd.read_csv(args.results)
    df = attach_e_components(df, gt)

    # Save augmented CSV for the thesis appendix.
    aug_path = out_dir / "results_with_components.csv"
    df.to_csv(aug_path, index=False)
    print(f"Wrote augmented results -> {aug_path}")

    # ---- Table: E1-E4 breakdown ----
    breakdown = df.groupby("strategy")[
        ["e1_precision", "e2_uncertainty", "e3_traceability", "e4_anomaly_flag", "astro_explain"]
    ].mean().round(3)
    breakdown = breakdown.loc[["react", "plan_execute", "reflexion"]]
    print("\n=== AstroExplain component breakdown (mean) ===")
    print(breakdown.to_string())

    # ---- Figure: components bar ----
    fig_components_bar(df, out_dir / "fig_explain_components.png")
    print(f"Wrote -> {out_dir / 'fig_explain_components.png'}")

    # ---- Figure: T3 F1 box plot ----
    fig_t3_box(df, out_dir / "fig_t3_box.png")
    print(f"Wrote -> {out_dir / 'fig_t3_box.png'}")

    # ---- Figure: scatter anomaly_count vs T3 F1 ----
    pearson, spearman = fig_scatter_anomaly(df, gt, out_dir / "fig_scatter_anomaly.png")
    print(f"Wrote -> {out_dir / 'fig_scatter_anomaly.png'}  (r={pearson:.3f}, rho={spearman:.3f})")

    # ---- Figure: ablation curve ----
    ablation_values: dict[int, float] = {}
    refl_t3 = df[(df["strategy"] == "reflexion") & (df["task_type"] == "t3_anomaly")]
    ablation_values[2] = float(refl_t3["t3_f1"].mean())
    for pair in args.ablation:
        try:
            k_str, csv_path = pair.split(":", 1)
            k = int(k_str)
            sub_df = pd.read_csv(csv_path)
            sub_t3 = sub_df[(sub_df["strategy"] == "reflexion") & (sub_df["task_type"] == "t3_anomaly")]
            ablation_values[k] = float(sub_t3["t3_f1"].mean())
        except Exception as e:
            print(f"Skipping ablation pair {pair!r}: {e}")
    if len(ablation_values) >= 2:
        fig_ablation_curve(ablation_values, out_dir / "fig_ablation_max.png")
        print(f"Wrote -> {out_dir / 'fig_ablation_max.png'}")
        print(f"  Ablation values: {ablation_values}")
    else:
        print("(Skipping ablation curve — need at least 2 points)")


if __name__ == "__main__":
    main()

## §11 Optional — LLM-as-judge (`eval/llm_judge.py`)

Not used for headline numbers. Useful for spot-checking edge cases where the deterministic AstroExplain rubric seems off.

In [ ]:
"""
Optional LLM-as-judge for the AstroExplain rubric.

The primary `BenchmarkMetrics.astro_explain_score` is deterministic and
fast — that's the headline result. This module provides an *optional*
LLM-graded version for sanity-checking the rubric: useful when reviewing
edge-case answers where the deterministic score seems off.
"""

from __future__ import annotations

import json
import re
from dataclasses import dataclass


JUDGE_PROMPT = """You are evaluating an astronomical-data-analysis answer
against a 4-criterion rubric. Score each criterion as 0 or 1 (integer).

Criteria:
- E1 (precision): Does the answer reference specific HDU indices or FITS
  header keywords (e.g. EXPTIME, NAXIS, BITPIX, TELESCOP)?
- E2 (uncertainty): If the file has high NaN ratio (>10%) or quality
  issues, does the answer hedge using cautious language?
- E3 (traceability): Does the answer cite specific numerical values that
  trace back to the tool outputs provided below?
- E4 (anomaly_flag): If the file has anomalies, does the answer mention
  them explicitly?

Reply with valid JSON only:
{{"e1": 0|1, "e2": 0|1, "e3": 0|1, "e4": 0|1, "rationale": "..."}}

ANSWER TO JUDGE:
{answer}

TOOL OUTPUTS:
{tool_outputs}

FILE HAS ANOMALY: {has_anomaly}
NAN RATIO: {nan_ratio}
"""


@dataclass
class LLMJudgeScore:
    e1: int
    e2: int
    e3: int
    e4: int
    total: float  # /4 → in [0, 1] after scaling.
    rationale: str


class LLMJudge:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def score(
        self,
        answer: str,
        tool_calls: list[dict],
        has_anomaly: bool,
        nan_ratio: float | None,
    ) -> LLMJudgeScore:
        outputs_str = json.dumps(tool_calls, indent=2, default=str)[:8000]
        prompt = JUDGE_PROMPT.format(
            answer=answer or "",
            tool_outputs=outputs_str,
            has_anomaly=has_anomaly,
            nan_ratio=nan_ratio if nan_ratio is not None else "unknown",
        )
        resp = self.llm.complete(
            [
                {"role": "system", "content": "You are a strict evaluator. Reply JSON only."},
                {"role": "user", "content": prompt},
            ]
        )
        data = _parse_judge_json(resp.content)
        e1, e2, e3, e4 = (int(data.get(k, 0)) for k in ("e1", "e2", "e3", "e4"))
        total = (e1 + e2 + e3 + e4) / 4.0
        return LLMJudgeScore(
            e1=e1, e2=e2, e3=e3, e4=e4,
            total=total,
            rationale=str(data.get("rationale", "")),
        )


def _parse_judge_json(text: str) -> dict:
    if not text:
        return {}
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return {}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return {}

## §12 Run the pipeline (optional — takes ~1h on Groq)

Skip this section if you only want to inspect the cached results.

```
# 1. Download (~30 min, 1.9 GB) — produces dataset/fits_files/ + manifest.json
download_dataset(output_dir='dataset/fits_files', n_normal=30, n_anomaly=10)

# 2. Generate ground truth (~1 min) — produces ground_truth.json
gen = GroundTruthGenerator()
gts = gen.generate_batch('dataset/fits_files')
gen.save(gts, 'ground_truth.json')

# 3. Run benchmark (~30-60 min, 3 strategies × 3 tasks × 40 files = 360 LLM rounds)
harness = BenchmarkHarness()
results = harness.run(
    dataset_dir='dataset/fits_files',
    ground_truth_path='ground_truth.json',
    output_dir='fits_agent_eval_run/',
)
```


## §13 Final result — strategy comparison table

The cached comparison lives in `fits_agent_eval_results/comparison.md`. One table, three strategies, the metrics the report needs.

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
display(Markdown((Path('fits_agent_eval_results') / 'comparison.md').read_text(encoding='utf-8')))

## §14 Dataset reference

The benchmark ran on the dataset described by `ground_truth.json` (deterministic per-file answers for T1/T2/T3) and `manifest.json` (MAST file list with anomaly labels). Both kept in `fits_agent_eval_results/` for reproducibility.

In [ ]:
import json
gt = json.loads((Path('fits_agent_eval_results') / 'datasets' / 'ground_truth.json').read_text(encoding='utf-8'))
mf = json.loads((Path('fits_agent_eval_results') / 'datasets' / 'manifest.json').read_text(encoding='utf-8'))
n_anom = sum(1 for f in mf['files'] if f.get('expected_anomaly'))
print(f'Dataset: {len(gt)} ground-truth records  ·  {len(mf["files"])} files manifested  ·  {n_anom} flagged as anomaly')